# Imports

In [1]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.coordinates import FK5
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
# import healpy as hp
import matplotlib.pyplot as plt
from astropy.coordinates import Angle
from astropy import units
from scipy.interpolate import RectBivariateSpline
from scipy import interpolate
import copy
import os
import sys
from tqdm.notebook import tqdm
from tqdm import tqdm
import glob
import h5py
import matplotlib
# from src2.ms_fit_joint import *
from src2.fit_funcs import *
matplotlib.rcParams['mathtext.fontset'] = 'cm'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams["font.size"] = "18"
PATH="../Sensitivty_files/Sky_spectrum/"
import scipy.io

## Load necessary files and store constants

In [2]:
sellon=-140.625
sellat=-18.209956864283015
power_output=np.load('../Sensitivty_files/power_output.npy',allow_pickle=True)

In [3]:
CHANNEL_WIDTH       = 0.06103515625/2.0 #/* MHz */
NUMBER_OF_CHANNELS  = 8193
START_FREQUENCY     = 0.00 #/* MHz */

PCHANNEL_WIDTH      = 16.0*0.06103515625 #/* MHz */
PNUMBER_OF_CHANNELS = 41
PSTART_FREQUENCY    = 210.44921875 #/* MHz */

RCHANNEL_WIDTH      = 16.0*0.06103515625 #/* MHz */
RNUMBER_OF_CHANNELS = 103
RSTART_FREQUENCY    = 109.86328125 #/* MHz */

QCHANNEL_WIDTH      = 16.0*0.06103515625 #/* MHz */
QNUMBER_OF_CHANNELS = 72
QSTART_FREQUENCY    = 39.5507812500 #/* MHz */

SITE_LATITUDE       = sellat
SITE_LONGITUDE      = sellon
NHPIX               = 3072
PI                  = 3.14159265358979

TCMB                = 2.72548

filename_coord      = "PIXEL_LISTING_NESTED_R4_GALACTIC.txt" #GMOSS

filename_pix_spec1  ="total_model_spec_13jan17_3072pix_103fq.txt"
filename_pix_spec2  ="total_model_spec_24jan17_1_3072_freq_40_110_MHz_16spacing.txt"
filename_pix_spec3  ="total_model_spec_31jan17_210_250_16spacing.txt"

beam_path  = '/home/saurabhs/Documents/Yogen_starfire/Farfield1MHz//'

rt_file    = 'gamma_linear.txt'
file_list  = sorted(glob.glob(os.path.join(beam_path,"*.txt")))


##### Fitting constants

fmin = 56.0/1e3
fmax = 109.0/1e3

LOWF   = fmin*1e3
HGHF   = fmax*1e3


In [4]:
def Read_Two_Column_File(file_name):
    with open(file_name, 'r') as data:
        x = []
        y = []
        for val, line in enumerate(data):
            if val==0:
                continue
            p = line.split()
            x.append(float(p[0]))
            y.append(float(p[1]))
    return x, y

def Read_pixel_freq(file_name, nfreq):
    with open(file_name, 'r') as data:
        x = np.zeros((NHPIX, nfreq))
        for val, line in enumerate(data):
            if val==0:
                continue
            x[val-1] = [float(i) for i in line.split()]
    return x, np.shape(x)[1]

def select_freq_1d(x1, low, high): 
    prl = 0 
    prh = 0 
    for i in range(0, len(x1)):
        if x1[i]<=low:
            i_low=i+1
            prl = 1 
        if x1[i]>=high:
            i_high=i
            prh = 1 
            break
    if prl==1 and prh==0:
        i_high = len(x1)-1
    if prl==0 and prh==1:
        i_low = 1    
    return i_low, i_high


In [5]:
beam_3D_unnorm.shape

NameError: name 'beam_3D_unnorm' is not defined

In [6]:
def get_freq_from_file(filename):
    _temp = os.path.basename(filename).replace('.txt','').replace('mhz','')
    return float(_temp)

## GMOSS

In [7]:
ll_coordinate, bb_coordinate = np.radians(Read_Two_Column_File(PATH+filename_coord))

T1, nspec1 = Read_pixel_freq(PATH+filename_pix_spec1, RNUMBER_OF_CHANNELS)
T2, nspec2 = Read_pixel_freq(PATH+filename_pix_spec2, QNUMBER_OF_CHANNELS)
T3, nspec3 = Read_pixel_freq(PATH+filename_pix_spec3, PNUMBER_OF_CHANNELS)

freq    = np.zeros(nspec1+nspec2+nspec3)
T_pix   = np.zeros((NHPIX, nspec1+nspec2+nspec3))
print(np.shape(T_pix))

for i in range(0, nspec2):
    freq[i]=(float)(QSTART_FREQUENCY/1000.0) +\
            ((float)(QCHANNEL_WIDTH)/1000.0)*(float)(i)
for i in range(nspec2, (nspec1+nspec2)):
    freq[i]=(float)(RSTART_FREQUENCY/1000.0) + ((float)(RCHANNEL_WIDTH)/1000.0)*(float)(i-nspec2)
for i in range(nspec1+nspec2, (nspec1+nspec2+nspec3)):
    freq[i]=(float)(PSTART_FREQUENCY/1000.0)+((float)(PCHANNEL_WIDTH)/1000.0)*(float)(i-(nspec1+nspec2))

for j in range(0, NHPIX):
    T_pix[j][0:nspec2]=T2[j]
    T_pix[j][(nspec2):(nspec1+nspec2)]=T1[j]
    T_pix[j][(nspec1+nspec2):(nspec1+nspec2+nspec3)]=T3[j]
    
freq_org, T_pix_org = copy.deepcopy(freq), copy.deepcopy(T_pix)


(3072, 216)


In [8]:
ifr_low, ifr_hgh = select_freq_1d(freq_org, fmin, fmax)
freq  = freq_org[ifr_low:ifr_hgh]*1e3
T_pix = T_pix_org[:,ifr_low:ifr_hgh]

In [9]:
data21   = scipy.io.loadmat('../Sensitivty_files/Sky_spectrum/Data_18March_wMFP.mat')
data21   = data21['Data2']/1e3 
fr       = np.loadtxt('../Sensitivty_files/Sky_spectrum/freq_saras.txt') #Frequency in MHz
signal   = interpolate.interp1d(fr, data21)

# Convolve sky + RFI with beam pattern

In [10]:
# NPIX = hp.nside2npix(16) # Storing the number of pixels of the map corresponding to the given NSIDE
# pix=hp.ang2pix(16,sellon,sellat,lonlat=True)

# NPIX = 3072 # Storing the number of pixels of the map corresponding to the given NSIDE
# pix=hp.ang2pix(16,sellon,sellat,lonlat=True)

In [11]:
dt = TimeDelta(np.linspace(0.,24.*3600, 5), format='sec') #Will take ~5 min

obstimes = Time('2019-4-12 23:00:00') + dt

hanle = EarthLocation(lat=SITE_LATITUDE*u.deg, lon=SITE_LONGITUDE*u.deg, height=4500*u.m)
gc    = SkyCoord(l=ll_coordinate*u.radian, b=bb_coordinate*u.radian,\
        frame='galactic')

#Get RA, DEC 
trans      = gc.transform_to('icrs')
ra, dec    = trans.ra.hour, trans.dec.hour

observing_time = Time(obstimes, scale='utc', location=hanle)
LST            = observing_time.sidereal_time('mean').value

T_monopole_AUS_sin = np.zeros((obstimes.shape[0], freq.shape[0]))

nside=16
NPIX = 3072 # Storing the number of pixels of the map corresponding to the given NSIDE
arr=np.arange(NPIX)
# elev_ang=np.load('/home/saurabhs/Documents/Yogen_starfire/skymodels/elev_angle.npy',allow_pickle=True)
freq_rfi=np.arange(55,110,1)
freq1=freq_rfi
pix=1991

for val, (time,i) in enumerate(zip(obstimes, tqdm(range(len(obstimes))))):
    
    print("Processing time: {}".format(time))
   
    #Get Alt, Az
    
    trans_local                 = gc.transform_to(AltAz(obstime=time, location=hanle))
    az, alt                     = trans_local.az.degree, trans_local.alt.degree
   
    ind_below_horizon           = alt < 0
       
    ############################
    
    beam_gen = np.zeros_like(T_pix)

    for ifreq, freq_value in enumerate(freq):
        rogue_phi  = []
        for iangle, (alt_value, az_value) in enumerate(zip(alt, az)):
            
            if az_value > phi_array_new.max():
                rogue_phi.append(az_value)
                az_value = 360 - az_value
                
            beam_gen[iangle,ifreq] = np.cos(alt_angle)**2
    beam_gen[ind_below_horizon,:] = np.nan   
                                        
    T_monopole_AUS_sin[val]                      = (np.nansum(T_pix * beam_gen + power_output[0,pix,:],axis=0)/np.nansum(beam_gen,axis=0))
                                                                                    
    #############################


plt.plot(freq,T_monopole_AUS_sin[0])
plt.xlabel('Frequency [in MHz]',fontsize=13)
plt.ylabel('The Brightness Temperature [in Kelvin]',fontsize=13)
plt.yscale('log')



  0%|          | 0/5 [00:00<?, ?it/s]

Processing time: 2019-04-12 23:00:00.000


NameError: name 'phi_array_new' is not defined

In [ ]:
print(beam_gen[:,0])

## Add system noise

In [ ]:
tau=158.69*3600
sigma=np.zeros((5,55))
for i in range(5):
    for j in range(55):
        sigma[i,j]=(70+T_monopole_AUS_sin[i,j])/(np.sqrt((1e6)*tau))
        T_monopole_AUS_sin[i,j]+=np.random.normal(0,1,1)*sigma[i,j]

## Residuals for null hypothesis

In [ ]:
freq_null=copy.deepcopy(freq)

In [12]:
avg_spectra = (np.mean(T_monopole_AUS_sin, axis=0))
# Add radiometer noise

flags=avg_spectra>1e4
avg_spectra=avg_spectra[~flags]
freq_null=freq_null[~flags]
freq_null = np.asarray(freq_null, dtype=np.float64)
avg_spectra = np.asarray(avg_spectra, dtype=np.float64)
e=sigma[0,~flags]

# Codes taken from investigation_cleaner.ipynb

for i in range(3,11):
    print(i)
    initial_ = np.zeros(i, dtype='float128')
    if i ==3:
        initial_ = np.polyfit(y= np.log10(avg_spectra), x = rescale(freq_null), deg = 2)
    if i > 3:
        if para_new[0] ==0:
            p_order = len(para_new)-2
            break
        initial_[1:] = np.array(para_new, dtype='float128')
    para_new,res_cons,fit_curve_mag, chi_mag = chi_dunk_DC(x1=freq_null,y1=avg_spectra,\
                                             p00=initial_, gloss = False, keju_inf=0,\
                                               e1=e,iterations=50)
    

NameError: name 'freq_null' is not defined

In [13]:
rms_null = np.sqrt(np.mean(res_cons**2))
print(rms_null)

NameError: name 'res_cons' is not defined

In [14]:
np.save('../../no_sigTa',avg_spectra)
np.save('../../sig_Ta',avg_spectra1)

NameError: name 'avg_spectra1' is not defined

## Inject 21 cm signal

In [ ]:
data21   = scipy.io.loadmat('../Sensitivty_files/Sky_spectrum/Data_18March_wMFP.mat')
data21   = data21['Data2']/1e3 
fr       = np.loadtxt('../Sensitivty_files/Sky_spectrum/freq_saras.txt') #Frequency in MHz
signal   = interpolate.interp1d(fr, data21)
T21=  signal(freq)[100] #max 263 
freq_alt=copy.deepcopy(freq)

In [ ]:
avg_spectra1 = (np.mean(T_monopole_AUS_sin, axis=0)+T21)
flags=avg_spectra1>1e4

avg_spectra1=avg_spectra1[~flags]
freq_alt=freq_alt[~flags]
freq_alt = np.asarray(freq_alt, dtype=np.float64)
avg_spectra1 = np.asarray(avg_spectra1, dtype=np.float64)
e=sigma[0,~flags]

# Codes taken from investigation_cleaner.ipynb

for i in range(3,11):
    print(i)
    initial_ = np.zeros(i, dtype='float128')
    if i ==3:
        initial_ = np.polyfit(y= np.log10(avg_spectra1), x = (freq_alt), deg = 2)
    if i > 3:
        if para_new1[0] ==0:
            p_order = len(para_new1)-2
            break
        initial_[1:] = np.array(para_new1, dtype='float128')
    para_new1,res_cons1,fit_curve_mag1, chi_mag1 = chi_dunk_DC(x1=freq_alt,y1=avg_spectra1,\
                                             p00=initial_, gloss = True, keju_inf=0,\
                                               e1=e,iterations=50)
    #fit_curves_array = np.vstack((fit_curves_array,fit_curve_))
    #print(np.std(res_ns))

In [ ]:
excess_rms = np.sqrt(np.std(res_cons1)**2-np.std(res_cons)**2)
print(excess_rms/np.std(res_cons))

In [ ]:
# plt.plot(freq_null, (res_cons)*1000, alpha=0.6,label="No signal")
plt.plot(freq_null, (res_cons1)*1000, alpha=0.6,label="With signal")
# plt.plot(freq,T21*1000)
plt.legend()
plt.ylabel('residuals (mK)')
plt.xlabel('Frequency (MHz)')
plt.show()